# Generate Side-By-Side Comparison Plots
This notebook generates the side-by-side K-Means vs Ground Truth comparison plots.

### Required Files:
Ensure you have the following uploaded to the working directory in Colab (or extract your project ZIP file directly into `/content/`):
1. **`filtered_poems.csv`** (The main dataset)
2. **`embeddings/filtered_embeddings/`** folder containing your `.npy` model embeddings (e.g., `d2v_poem_d100_w5_dbow.npy`).
3. **`cluster_assignments/`** folder containing the CSV cluster output generated from `cluster_analysis.py` (e.g., `d2v_poem_d100_w5_dbow_Period_clusters.csv`).

In [ ]:
!pip install umap-learn pandas scikit-learn numpy scipy matplotlib seaborn

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.pipeline import make_pipeline
import umap
from scipy.sparse import load_npz, issparse

# Config
N_COMPONENTS = 50
UMAP_NEIGHBORS = 50
UMAP_MIN_DIST = 0.1

MODELS = {
    "d2v_poem_d100_w5_dbow": "d2v_poem_d100_w5_dbow",
    "d2v_line_d100_w5_dbow": "d2v_line_d100_w5_dbow",
    "w2v_d100_w5_sg_idf": "w2v_d100_w5_sg_idf",
}

TASKS = ["Emotion", "Format", "Period", "PoetTop20"]

# Setup paths
embeddings_dir = Path("embeddings/filtered_embeddings") if Path("embeddings/filtered_embeddings").exists() else Path("embeddings")
metadata_dir = Path("metadata") if Path("metadata").exists() else Path("cluster_assignments")
base_metadata = Path("metadata/filtered_poems.csv") if Path("metadata/filtered_poems.csv").exists() else Path("filtered_poems.csv")

output_dir = Path("plots/side_by_side")
output_dir.mkdir(parents=True, exist_ok=True)

# Orderings
PERIOD_ORDER = ["1971-Present", "1941-1970", "1901-1940", "1781-1900", "1550-1780", "Pre-1550", "Unknown"]

def get_sorting_key(x):
    """Sort strings handling cluster numbers, 'Unknown', 'N/A'"""
    x_str = str(x)
    if "Unknown" in x_str: return (2, x_str)
    if "N/A" in x_str or "-1" in x_str: return (3, x_str)
    if "Other" in x_str: return (1, x_str)
    return (0, x_str)

def sort_period_key(x):
    x_str = str(x)
    for i, p in enumerate(PERIOD_ORDER):
        if p in x_str:
            return (i, x_str)
    return (len(PERIOD_ORDER), x_str)

df_base = pd.read_csv(base_metadata)
df_base["Period"] = df_base.get("Period", "Unknown").fillna("Unknown")
df_base["Emotion"] = df_base.get("Emotion", "Unknown").fillna("Unknown").replace("", "Unknown")
df_base["Format"] = df_base.get("Format", "Unknown").fillna("Unknown")
if "Poet" in df_base.columns:
    top_poets = df_base["Poet"].value_counts().nlargest(20).index
    df_base["Plot_Poet"] = df_base["Poet"].where(df_base["Poet"].isin(top_poets), other="Other")
else:
    df_base["Plot_Poet"] = "Unknown"

for model_stem in MODELS:
    print(f"Loading {model_stem}...")
    vec_path = embeddings_dir / f"{model_stem}.npy"
    if not vec_path.exists():
        vec_path = Path("metadata") / f"{model_stem}.npy"
        if not vec_path.exists():
           for prefix in ["top1_", "top2_", "top3_"]:
              if (embeddings_dir / f"{prefix}{model_stem}.npy").exists():
                 vec_path = embeddings_dir / f"{prefix}{model_stem}.npy"
                 break

    if not vec_path.exists():
        print(f"Skipping {model_stem}, no vectors found.")
        continue

    if vec_path.suffix == ".npz":
        vectors = load_npz(vec_path)
    else:
        v_data = np.load(vec_path, allow_pickle=True)
        vectors = v_data["vectors"] if isinstance(v_data, np.lib.npyio.NpzFile) and "vectors" in v_data else v_data

    # Reduce
    if issparse(vectors):
        svd = make_pipeline(TruncatedSVD(n_components=N_COMPONENTS, random_state=42), Normalizer(copy=False))
        intermediate = svd.fit_transform(vectors)
    else:
        scaled = StandardScaler().fit_transform(vectors)
        n_comp = min(N_COMPONENTS, scaled.shape[1])
        pca = PCA(n_components=n_comp, random_state=42)
        intermediate = pca.fit_transform(scaled)

    print(f"Computing UMAP for {model_stem}...")
    reducer = umap.UMAP(n_components=2, n_neighbors=UMAP_NEIGHBORS, min_dist=UMAP_MIN_DIST, metric="cosine", random_state=42)
    coords_2d = reducer.fit_transform(intermediate)

    # Load specific cluster assignments
    df = df_base.copy()
    df["umap_x"] = coords_2d[:, 0]
    df["umap_y"] = coords_2d[:, 1]
    available_tasks = []

    for task in TASKS:
        tc_file = Path("cluster_assignments") / f"{model_stem}_{task}_clusters.csv"
        if not tc_file.exists():
            tc_file = Path("metadata") / f"{model_stem}_{task}_clusters.csv"

        if tc_file.exists():
            tc_df = pd.read_csv(tc_file)
            if "cluster_id" in tc_df.columns:
                df[f"cluster_id_{task}"] = tc_df["cluster_id"]
                available_tasks.append(task)

    task_targets = {
        "Emotion": "Emotion",
        "Format": "Format",
        "Period": "Period",
        "PoetTop20": "Plot_Poet"
    }

    # Generate Cluster Mappings
    for task in available_tasks:
        task_col = f"cluster_id_{task}"
        str_clusters = df[task_col].fillna(-1).astype(int).astype(str)
        target_col = task_targets.get(task)

        if target_col and target_col in df.columns:
            mapping = {}
            #Define the dominant attribute behind the cluster
            for cid in str_clusters.unique():
                if cid == "-1":
                    mapping[cid] = "N/A"
                else:
                    mode_s = df.loc[str_clusters == cid, target_col].mode()
                    mode_val = mode_s.iloc[0] if not mode_s.empty else "Unknown"
                    mapping[cid] = f"{mode_val} (C{cid})"
            df[f"Plot_Cluster_{task}"] = str_clusters.map(mapping)
        else:
            df[f"Plot_Cluster_{task}"] = str_clusters

    # Plotting
    for task in available_tasks:
        print(f"Plotting {model_stem} Task: {task}")
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        fig.suptitle(f"UMAP Projection (n={UMAP_NEIGHBORS}, d={UMAP_MIN_DIST}) - {model_stem}\nTask: {task}", fontsize=16)

        gt_col = task_targets[task]
        pred_col = f"Plot_Cluster_{task}"

        # Determine palettes
        palette_gt = "viridis" if task == "Period" else "Set3" if task == "PoetTop20" else "Set2"
        palette_pred = "Dark2"

        # Sort values so legends are consistent
        sort_func = sort_period_key if task == "Period" else get_sorting_key

        gt_unique = sorted(df[gt_col].unique(), key=sort_func)
        pred_unique = sorted(df[pred_col].unique(), key=sort_func)

        # Create explicit color dictionaries, assigning light gray to unclustered data
        gt_colors = sns.color_palette(palette_gt, n_colors=len(gt_unique))
        gt_cmap = {val: gt_colors[i] for i, val in enumerate(gt_unique)}
        for key in ["Other", "Unknown", "N/A"]:
            if key in gt_cmap: gt_cmap[key] = "#d3d3d3"

        pred_colors = sns.color_palette(palette_pred, n_colors=len(pred_unique))
        pred_cmap = {val: pred_colors[i] for i, val in enumerate(pred_unique)}
        for key in ["Other", "Unknown", "N/A"]:
            if key in pred_cmap: pred_cmap[key] = "#e0e0e0"

        # Plot ordering: ensure "Other", "Unknown", "N/A" are plotted first (bottom layer)
        df["_gt_order"] = df[gt_col].apply(lambda x: 0 if x in ["Other", "Unknown", "N/A"] else 1)
        df_gt = df.sort_values("_gt_order")

        df["_pred_order"] = df[pred_col].apply(lambda x: 0 if x in ["Other", "Unknown", "N/A"] else 1)
        df_pred = df.sort_values("_pred_order")

        sns.scatterplot(data=df_gt, x="umap_x", y="umap_y", hue=gt_col, palette=gt_cmap, s=15, alpha=0.7, edgecolor=None, hue_order=gt_unique, ax=axes[0])
        axes[0].set_title(f"Ground Truth ({gt_col})")
        axes[0].set_xticks([]); axes[0].set_yticks([])
        axes[0].legend(bbox_to_anchor=(0, -0.05), loc='upper left', title=gt_col, fontsize='small', ncol=2 if len(gt_unique)>10 else 1)

        sns.scatterplot(data=df_pred, x="umap_x", y="umap_y", hue=pred_col, palette=pred_cmap, s=15, alpha=0.7, edgecolor=None, hue_order=pred_unique, ax=axes[1])
        axes[1].set_title(f"K-Means Clusters")
        axes[1].set_xticks([]); axes[1].set_yticks([])
        axes[1].legend(bbox_to_anchor=(0, -0.05), loc='upper left', title="Cluster (Dominant)", fontsize='small', ncol=2 if len(pred_unique)>10 else 1)

        plt.tight_layout()
        plt.savefig(output_dir / f"umap_n{UMAP_NEIGHBORS}_d{UMAP_MIN_DIST}_{model_stem}_{task}_comparison.png", bbox_inches='tight')
        plt.close(fig)

print("Finished generating all plots!")

